# Homework: Breaking the Sequence Model
## The Challenge:
In class, we built an LSTM to classify movie reviews, and I made two big theoretical claims:

1. Standard RNNs have amnesia. They suffer from the Vanishing Gradient problem and forget early words.
2. We shouldn't delete stop words. Unlike old Machine Learning, Deep Learning needs words like "not" to understand context.

Your homework is to experimentally prove (or disprove!) both of these claims by modifying the code we wrote in class.

## Task 1: The Vanilla RNN Showdown
PyTorch makes it incredibly easy to swap out neural network architectures.
Below is the SentimentLSTM we built in class. Your task is to write a new class called SentimentRNN that uses PyTorch's standard nn.RNN module instead of nn.LSTM.

Hint: The standard nn.RNN only outputs the hidden state, it does not have a cell state (the conveyor belt). You will need to adjust the forward function so it doesn't crash trying to unpack a cell state that doesn't exist!

In [ ]:
import torch
import torch.nn as nn

class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, text):
        embedded = self.embedding(text)
        lstm_out, (hidden, cell) = self.lstm(embedded)
        final_hidden_state = hidden[-1]
        out = self.fc(final_hidden_state)
        return self.sigmoid(out).squeeze()

class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, text):
        embedded = self.embedding(text)
        rnn_out, hidden = self.rnn(embedded)
        final_hidden_state = hidden[-1]
        out = self.fc(final_hidden_state)
        return self.sigmoid(out).squeeze()

### Thought Experiment:
If you train both of these models on the IMDB dataset for 5 epochs, the SentimentLSTM will likely reach ~85% accuracy, while the SentimentRNN might struggle to pass 60%. Based on the lecture, explain mathematically why the vanilla RNN is failing to classify a 200-word movie review accurately.

### Analysis:

The vanila RNN struggles because gradients are vanishing. They become smaller and smaller and this causes the model to forget the initial context.

## Task 2: The "Old Rules" Experiment
In class, I told you that deleting "Stop Words" (the, is, not, at) is a bad idea for Deep Learning sequence models. Let's see why.

Write a basic Python function that takes a movie review and removes the stop words. Then, look at the output and explain why feeding this "cleaned" text to an LSTM might cause it to make the wrong prediction.

In [ ]:
stop_words = ["i", "me", "my", "the", "and", "is", "in", "it", "not", "was", "a", "an", "to"]

def remove_stop_words(review):
    words = review.split()
    filtered_words = [word for word in words if word not in stop_words]
    return " ".join(filtered_words)

review = "i thought the movie was not bad and it was a joy to watch"
cleaned_review = remove_stop_words(review)

print(f"Original: {review}")
print(f"Cleaned:  {cleaned_review}")

Original: i thought the movie was not bad and it was a joy to watch
Cleaned:  thought movie bad joy watch


### Analysis:
If you pass this cleaned string into our SentimentLSTM, what do you think it will predict? Why is this a problem?

### Analysis:

If we pass the `cleaned_review`: `"thought movie bad joy watch"` into our `SentimentLSTM`, it would most likely predict a **Negative** sentiment.

This is a problem because the original review, `"i thought the movie was not bad and it was a joy to watch"`, is positive.

This demonstrates why removing stop words can be bad for Deep Learning sequence models like LSTMs.